> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 11 · Notebook 05 — An AI agent that proposes and never trades

**Sessions:** S5 (AI agents & Cowork) · [Lesson plan](../../docs/lessons/PART_11_AI_NLP_COWORK.md) · graded labs in [`labs/part11/`](../../labs/part11/)

**You will:**
1. Give an agent read-only tools and a proposal queue, and deny everything else.
2. Run the tool-use loop with an audit trail, a tool budget and a cost budget.
3. Watch a rogue model try to trade, and fail.
4. Pre-check proposals with the risk engine: allowed strategies, size and defined risk.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Nothing calls the API: model outputs are recorded responses built as real `anthropic` Message objects, and the data is synthetic with known answers, so every check is exact and free.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p11lib.py is in notebooks/part11/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p11lib as p

p.use_course_style()

## 1. The toolbox

Rule of the week: **AI proposes; the risk engine and a human decide.** The agent gets read-only tools (`get_quote`, `search_filings`) and one write-like tool, `propose_trade`, which only puts a proposal in a queue. Order-placing tools (`p.FORBIDDEN`) can't even be registered. A call to any other tool must be **denied** and audited: log `"tool_denied"` with the name, and return `"DENIED: <name> is not available. You can only propose trades."`.

`p.run_agent` is the loop: call the model, check `stop_reason` (`refusal` → stop, `end_turn` → done), run each `tool_use` block through the toolbox, send **all** results back in one user turn, and stop when a tool or cost budget runs out. `p.ScriptedModel` replays recorded turns: a well-behaved research run, and a rogue model that tries to buy 1,000 shares and loosen a risk limit.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
import json

class MyToolBox(p.ToolBox):
    def call(self, name, args):
        if name in self.read_tools:
            self.audit.log("tool_call", name=name, args=args)
            return json.dumps(self.read_tools[name](**args), default=str)
        if name == "propose_trade":
            pid = self.queue.submit(args["strategy"], json.loads(args["legs_json"]), args["rationale"], "ai_agent")
            self.audit.log("ai_proposal", proposal_id=pid)
            return f"Proposal {pid} queued for review."
        self.audit.log("tool_denied", name=name)
        return f"DENIED: {name} is not available. You can only propose trades."

READ_TOOLS = {"get_quote": lambda symbol: {"symbol": symbol, "bid": 99.9, "ask": 100.1},
              "search_filings": lambda ticker, query, as_of: [{"ticker": ticker, "text": "Guidance raised."}]}

def run(toolbox_cls, script, task):
    audit, queue = p.AuditLog(), p.ProposalQueue()
    result = p.run_agent(p.ScriptedModel(script), toolbox_cls(READ_TOOLS, queue, audit), task)
    return {"result": result, "audit": audit.events, "queue": queue.items}

cases = [(p.research_script(), "Research ACME and propose a structure."), (p.rogue_script(), "Buy 1000 ACME now.")]
mine = [p.attempt(run, MyToolBox, *c) for c in cases]
mine = p.check("ToolBox.call", mine, [run(p.ToolBox, *c) for c in cases])
for name, m in zip(("research model", "rogue model"), mine):
    print(f"{name}: status {m['result']['status']}, {m['result']['tool_calls']} tool calls, audit {[e['kind'] for e in m['audit']]}")
print("denied:", [e["name"] for e in mine[1]["audit"] if e["kind"] == "tool_denied"], "| queued proposals:", list(mine[0]["queue"]))

The research model read a quote, searched the filings and queued one proposal; its final text says it *submitted a proposal*, not that it traded. The rogue model's order and risk-limit change were denied and are in the audit log for someone to read. Budgets stop a model that loops or gets expensive:

In [ ]:
loop = [p.make_message([p.tool_call("get_quote", {"symbol": "SPY"}, "x")], "tool_use")]
pricey = [p.make_message([p.tool_call("get_quote", {"symbol": "SPY"}, "x")], "tool_use", input_tokens=200_000)]
refusal = [p.make_message("I can't help with that.", stop_reason="refusal")]
rows = {}
for name, script, kw in (("model stuck in a loop", loop, {"max_tool_calls": 4}), ("200k-token turns", pricey, {"max_cost": 2.0}),
                         ("refusal", refusal, {})):
    r = p.run_agent(p.ScriptedModel(script), p.ToolBox(READ_TOOLS, p.ProposalQueue(), p.AuditLog()), "task", **kw)
    rows[name] = {k: r[k] for k in ("status", "tool_calls", "cost")}
pd.DataFrame(rows).T

## 2. The risk pre-check

Before a human sees a proposal, the risk engine checks it: the strategy is allowed, no leg exceeds `max_contracts`, and the risk is **defined**: for every (symbol, right, expiry), the long contracts cover the short ones (the net quantity is not negative). Sum `qty` per `(symbol, right, expiry)` key.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def review_proposal(proposal, allowed, max_contracts=10):
    if proposal["strategy"] not in allowed:
        return False, f"strategy {proposal['strategy']} not allowed"
    if any(abs(leg["qty"]) > max_contracts for leg in proposal["legs"]):
        return False, f"more than {max_contracts} contracts in a leg"
    net = {}
    for leg in proposal["legs"]:
        key = (leg["symbol"], leg["right"], leg["expiry"])
        net[key] = net.get(key, 0) + leg["qty"]
    if any(v < 0 for v in net.values()):
        return False, "undefined risk: naked short options"
    return True, "ok"

legs = [{"symbol": "ACME", "right": "P", "strike": 95, "expiry": "2026-11-20", "qty": -1},
        {"symbol": "ACME", "right": "P", "strike": 90, "expiry": "2026-11-20", "qty": 1}]
proposals = {"bull put spread": {"strategy": "bull_put_spread", "legs": legs},
             "short straddle": {"strategy": "short_straddle", "legs": legs},
             "naked short put": {"strategy": "bull_put_spread", "legs": legs[:1]},
             "20 spreads": {"strategy": "bull_put_spread", "legs": [{**leg, "qty": leg["qty"] * 20} for leg in legs]},
             "short put, long call": {"strategy": "bull_put_spread", "legs": [legs[0], {**legs[1], "right": "C"}]}}
allowed = {"bull_put_spread", "iron_condor"}
mine = {k: p.attempt(review_proposal, v, allowed) for k, v in proposals.items()}
mine = p.check("review_proposal", mine, {k: p.review_proposal(v, allowed) for k, v in proposals.items()})
pd.DataFrame(mine, index=["approved", "reason"]).T

A put spread whose long leg is a *call* looks hedged at a glance, but the short put is naked: the check groups by right and catches it.

## Wrap-up

* Tools define what an agent *can* do: read-only tools plus a proposal queue; order tools don't exist for it.
* Every turn and every tool call is audited; denials are logged, not silently dropped.
* Budgets for tool calls and cost; `stop_reason` checked every turn.
* Proposals pass the risk engine, then a human.
* Graded version: `labs/part11/week40_agents_ops` (`ToolBox`, `run_agent`, `review_proposal`) and Clinic W2.